# 06 · item2vec và vector phiên (ticket #15)

**Input** (Add Input): Kaggle Dataset `titeohihi/mixnext-splited` (output của notebook 03).

**Output** (`/kaggle/working/out/`):
- `candidates/item2vec_{val,test}_run0.parquet`
- `item_embeddings/item2vec.npy`, `item_embeddings/item2vec_tracks.parquet`
- `item2vec_params.json`: tham số đã chọn và NDCG@10 trên val của cả lưới (lần chạy 1, 2 ở #26 dùng lại file này qua `--params`)

Word2Vec skip-gram (gensim) trên chuỗi bài của mỗi playlist train; negative 5, epochs 5, mọi bài train đều có vector. Vector phiên và điểm gu giống ALS (#14, `src/models/session.py`).

Lưới chọn trên val: window {3, 10} × vector_size {128, 256} × decay {0,5; 0,7; 0,9; 1,0} (16 cấu hình, 4 lần học Word2Vec). Chọn theo NDCG@10.

Học nhiều luồng nên cùng `random_seed` không ra cùng vector từng bit; khác biệt giữa các lần chạy đo ở #26.

gensim có sẵn trên Kaggle (#2). Bật **Internet** để clone repo. Chỉ cần CPU. Ước tính 25–35 phút.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #15 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "b"
!rm -rf /kaggle/working/MusicRecSys && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys /kaggle/working/MusicRecSys
import sys; sys.path.insert(0, "/kaggle/working/MusicRecSys")

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("eval_targets.parquet"))[0].parent
OUT = Path("/kaggle/working/out")
print("input:", DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
import time
from src.models.item2vec import main

t0 = time.time()
report = main(["--data", str(DATA), "--out", str(OUT)])   # chọn tham số trên val, xuất file, có validate_dataset bên trong
print(f"{time.time() - t0:.0f} giây")
for p in sorted(OUT.rglob("*.*")):
    print(p.relative_to(OUT), f"{p.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Lưới trên val, xếp theo NDCG@10, so với baseline (#9) và ALS (#14), cùng 9.212 phiên val
import pandas as pd

REFERENCE_NDCG = {"popularity": 0.0040, "itemknn": 0.1297, "als": 0.0967}
grid = pd.DataFrame(report["grid"]).sort_values("ndcg", ascending=False)
print(grid.round(4).to_string(index=False))
ndcg = report["val"]["ndcg"]
print(f"\nChọn: {report['params']} | NDCG@10 val {ndcg:.4f}")
for name, ref in REFERENCE_NDCG.items():
    print(f"  so với {name} {ref}: {ndcg / ref - 1:+.1%}")

## Đăng output

1. **Save Version → Save & Run All (Commit)** để lưu output.
2. Tab Output của version vừa chạy → **New Dataset** tên `mixnext-item2vec`; chỉ chọn thư mục `out`, bỏ `MusicRecSys`.
   - Không thêm được vào `mixnext-embeddings`: Dataset tạo từ output notebook chỉ nhận version mới từ chính notebook đó (notebook 05).
   - Notebook sau Add Input bộ này và tìm file bằng `Path("/kaggle/input").rglob(<tên file>)`.

## Kết quả trên val (chưa chạy)

Dán bảng lưới và dòng "Chọn: …" vào đây sau khi chạy trên Kaggle.

Nếu window hoặc vector_size tốt nhất nằm ở mép lưới (window 3 hoặc 10, vector_size 256) thì mở rộng lưới về phía đó trong `GRID` của `src/models/item2vec.py` rồi chạy lại, như ALS ở notebook 05. Giới hạn: `item2vec.npy` phải đủ nhỏ để `artifacts/` của app dưới 200 MB (#20); 256 chiều đã khoảng 134 MB. decay giữ đúng tập của spec §6.